# Finding Optimal Lags
### Authors Andy Jiang & Collin McDevitt


In [32]:
%pip install neuralforecast yfinance matplotlib pandas numpy statsmodels pmdarima scikit-learn darts setuptools
%pip install --upgrade "packaging<24.0" setuptools

  Using cached packaging-26.0-py3-none-any.whl.metadata (3.3 kB)
Using cached packaging-26.0-py3-none-any.whl (74 kB)
  Attempting uninstall: packaging
    Found existing installation: packaging 23.2
    Uninstalling packaging-23.2:
      Successfully uninstalled packaging-23.2
Note: you may need to restart the kernel to use updated packages.
  Using cached packaging-23.2-py3-none-any.whl.metadata (3.2 kB)
Using cached packaging-23.2-py3-none-any.whl (53 kB)
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0
    Uninstalling packaging-26.0:
      Successfully uninstalled packaging-26.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ray 2.54.0 requires packaging>=24.2, but you have packaging 23.2 which is incompatible.
xarray 2026.2.0 requires packaging>=24.1, but you have packaging 23.2 which is incompatible.
Note: you may need t

In [33]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pandas_datareader import data as pdr

import yfinance as yf
from darts import TimeSeries

from darts.models import NBEATSModel, NHiTSModel

from darts.dataprocessing.transformers import Scaler

from darts.metrics import rmse
import pmdarima as pm
from sklearn.metrics import mean_squared_error

from fredapi import Fred


from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

import seaborn as sns


from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import TimeSeriesSplit


In [34]:
# Loading Data
df = pd.read_csv("dataframe.csv", parse_dates=["ds"]).sort_values("ds")
df = df.set_index("ds")


In [35]:

print("Shape:", df.shape)
print("Date range:", df.index.min(), "to", df.index.max())
print("Duplicate dates:", df.index.duplicated().sum())
print("Missing values:\n", df.isna().sum())


Shape: (5787, 10)
Date range: 2003-01-02 00:00:00 to 2025-12-31 00:00:00
Duplicate dates: 0
Missing values:
 y                     0
vix                   0
EPU                   0
UMCSENT               0
JOBLESS               0
FEDFUNDS              0
FLOWS                 0
CPI                   0
inflation_yoy         0
inflation_yoy_lag1    0
dtype: int64


In [ ]:
date_diffs = df.index.to_series().diff().value_counts().sort_index()
print("Date differences: \n", date_diffs.head(10))

Date differences: 
 ds
1 days    4531
2 days      55
3 days    1044
4 days     154
5 days       2
Name: count, dtype: int64


In [37]:

desc = df.describe().T
desc["median"] = df.median()
desc["skew"] = df.skew()
desc["kurtosis"] = df.kurtosis()
desc["missing"] = df.isna().sum()
print(desc[["count","missing","mean","median","std","min","25%","50%","75%","max","skew","kurtosis"]])



                     count  missing           mean         median  \
y                   5787.0        0     149.883666      85.314056   
vix                 5787.0        0      19.108573      16.799999   
EPU                 5787.0        0     123.440994      94.650000   
UMCSENT             5787.0        0      80.103093      81.200000   
JOBLESS             5787.0        0  370398.306549  316000.000000   
FEDFUNDS            5787.0        0       1.755562       1.010000   
FLOWS               5787.0        0      71.794510      35.430000   
CPI                 5787.0        0     240.896074     236.222000   
inflation_yoy       5787.0        0       0.083481       0.000000   
inflation_yoy_lag1  5787.0        0       0.083554       0.000000   

                              std            min           25%            50%  \
y                      149.077308      20.102070      38.30294      85.314056   
vix                      8.400969       9.140000      13.60500      16.799999 

In [38]:
df["y_roll_mean_21"] = df["y"].rolling(21).mean()
df["y_roll_std_21"] = df["y"].rolling(21).std()


print("\nCorrelation matrix:\n", df.corr(numeric_only=True))


max_lag = 10
lag_corrs = {}
for col in df.columns:
    if col != "y":
        lag_corrs[col] = {
            f"lag_{lag}": df["y"].corr(df[col].shift(lag))
            for lag in range(max_lag + 1)
        }

lag_corrs_df = pd.DataFrame(lag_corrs).T
print("\nLagged correlations with y:\n", lag_corrs_df)


Correlation matrix:
                            y       vix       EPU   UMCSENT   JOBLESS  \
y                   1.000000 -0.049753  0.404911 -0.391710 -0.068740   
vix                -0.049753  1.000000  0.405489 -0.477320  0.374333   
EPU                 0.404911  0.405489  1.000000 -0.456569  0.384056   
UMCSENT            -0.391710 -0.477320 -0.456569  1.000000 -0.101585   
JOBLESS            -0.068740  0.374333  0.384056 -0.101585  1.000000   
FEDFUNDS            0.376125 -0.235535 -0.006376 -0.109127 -0.231729   
FLOWS               0.306736  0.356730  0.601345 -0.307089  0.149287   
CPI                 0.936806 -0.022103  0.371317 -0.423455 -0.092331   
inflation_yoy       0.102814 -0.131802 -0.033385 -0.047624 -0.094324   
inflation_yoy_lag1  0.100606 -0.134299 -0.036380 -0.047598 -0.093857   
y_roll_mean_21      0.999069 -0.035881  0.407069 -0.392987 -0.074077   
y_roll_std_21       0.817667  0.198927  0.475179 -0.444651  0.076269   

                    FEDFUNDS     FLOWS   

We can see a lot of correlation, this is juts because everything rises. We need to look at $y$ percentage change.

In [42]:
df["y_log_ret"] = np.log(df["y"]).diff()
df["y_pct_change"] = df["y"].pct_change()

df.corr()


,y,vix,EPU,UMCSENT,JOBLESS,FEDFUNDS,FLOWS,CPI,inflation_yoy,inflation_yoy_lag1,y_roll_mean_21,y_roll_std_21,y_log_ret,y_pct_change,y_ret
y,1.000000,-0.049753,0.404911,-0.391710,-0.068740,0.376125,0.306736,0.936806,0.102814,0.100606,0.999069,0.817667,0.014551,0.014590,0.014590
vix,-0.049753,1.000000,0.405489,-0.477320,0.374333,-0.235535,0.356730,-0.022103,-0.131802,-0.134299,-0.035881,0.198927,-0.126752,-0.117542,-0.117542
EPU,0.404911,0.405489,1.000000,-0.456569,0.384056,-0.006376,0.601345,0.371317,-0.033385,-0.036380,0.407069,0.475179,0.030637,0.034160,0.034160
UMCSENT,-0.391710,-0.477320,-0.456569,1.000000,-0.101585,-0.109127,-0.307089,-0.423455,-0.047624,-0.047598,-0.392987,-0.444651,0.004219,0.001451,0.001451
JOBLESS,-0.068740,0.374333,0.384056,-0.101585,1.000000,-0.231729,0.149287,-0.092331,-0.094324,-0.093857,-0.074077,0.076269,0.044885,0.046803,0.046803
FEDFUNDS,0.376125,-0.235535,-0.006376,-0.109127,-0.231729,1.000000,0.130964,0.294263,0.045855,0.048234,0.374563,0.281234,-0.000923,-0.001712,-0.001712
FLOWS,0.306736,0.356730,0.601345,-0.307089,0.149287,0.130964,1.000000,0.278720,-0.049476,-0.051172,0.314665,0.453681,-0.015874,-0.010720,-0.010720
CPI,0.936806,-0.022103,0.371317,-0.423455,-0.092331,0.294263,0.278720,1.000000,0.075631,0.074679,0.938116,0.798187,0.005600,0.005984,0.005984
inflation_yoy,0.102814,-0.131802,-0.033385,-0.047624,-0.094324,0.045855,-0.049476,0.075631,1.000000,0.905901,0.102679,0.073553,0.013395,0.012067,0.012067
inflation_yoy_lag1,0.100606,-0.134299,-0.036380,-0.047598,-0.093857,0.048234,-0.051172,0.074679,0.905901,1.000000,0.100896,0.070624,-0.011963,-0.013596,-0.013596


In [ ]:
# future look at lags for each of the features